# Sentinel-2 y Landsat en el Biobío medio: imágenes, series temporales y anomalías

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Deadx7913/comparacion-sentinel2-landsat-biobio/blob/main/comparacion_sentinel2_landsat.ipynb)

Cuaderno híbrido basado en el taller *Datos FAIR, series temporales satelitales y detección de anomalías con Python*
(curso Geomática Aplicada, `alvaroparedesl/geomatica-aplicada`, carpeta `talleres/fair-series-temporales`). Junta en un
solo flujo los bloques **03** (acceso y visualización), **04** (serie temporal), **05** (exploración, climatología y
anomalías) y **07** (eventos extremos, opcional), y los aplica a **dos satélites a la vez** para ver en qué se diferencian.

**Zona:** Biobío medio entre Nacimiento y aguas arriba de Negrete, con las confluencias del Vergara, el Duqueco y el
Bureo (9 puntos, thalweg de secciones de un modelo hidráulico 1D).

| Bloque | Basado en | Qué hace |
|---|---|---|
| 0 | todos | Entorno, parámetros y puntos |
| A | 03 | Una imagen de cada satélite casi el mismo día: color verdadero, índices, temperatura, zoom y valor en los puntos |
| B | 04 | Serie temporal en los puntos con cada satélite (ventanas pequeñas, máscara de nubes, parquet con metadatos) |
| C | nuevo | Comparación directa: las fechas en que los dos satélites pasaron el mismo día |
| D | 05 | Exploración: estadísticos, medias móviles, ciclo anual, tendencia y anomalías (z-score) |
| E | 07 | Eventos extremos (opcional): climatología por ventana de días, episodios e Isolation Forest |

**Sentinel-2 frente a Landsat 8/9, en una tabla**

| | Sentinel-2 L2A | Landsat 8 y 9 C2 L2 |
|---|---|---|
| Píxel | 10 m (visible, NIR), 20 m (SWIR) | 30 m (ópticas), 100 m (térmica, entregada a 30 m) |
| Revisita | 2 a 5 días (2 o 3 satélites) | 8 días (2 satélites) |
| Desde | 2015 (L2A en Planetary Computer desde 2016-2017) | 2013 (Landsat 8) |
| Banda térmica | no | sí: temperatura de superficie |
| Paso a reflectancia | DN × 0,0001 + offset (0 o -0,1 según el baseline) | DN × 0,0000275 - 0,2 |
| Máscara de nubes | banda SCL (clases) | banda qa_pixel (bits) |

Se sigue el **patrón nativo** del curso: la búsqueda (`pystac_client`) y la carga (`odc.stac.load`) son llamadas
nativas, y los helpers de `utils/` del curso solo preparan sus argumentos y ordenan los datos.

## 0. Entorno, parámetros y puntos

La primera celda descarga el repositorio del curso para usar su carpeta `utils/`, fijado en la versión con la que se
escribió este cuaderno (`COMMIT_CURSO`). Si quieres la última versión del curso, deja `COMMIT_CURSO = None`.

In [ ]:
# @title Preparar entorno (ejecutar primero)
import subprocess
import sys
from pathlib import Path

REPO_CURSO = "https://github.com/alvaroparedesl/geomatica-aplicada"
COMMIT_CURSO = "4e77d8fa0e65e9efe3ae75e0ca30d85572816210"  # versión del curso con la que se escribió este cuaderno

EN_COLAB = "google.colab" in sys.modules
TRABAJO = Path("/content") if EN_COLAB else Path.cwd()
RAIZ = TRABAJO / "geomatica-aplicada"

if not (RAIZ / "utils").is_dir():
    RAIZ.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "init", "-q", str(RAIZ)], check=True)
    subprocess.run(["git", "-C", str(RAIZ), "remote", "add", "origin", REPO_CURSO], check=False)
    referencia = COMMIT_CURSO or "main"
    if subprocess.run(["git", "-C", str(RAIZ), "fetch", "-q", "--depth", "1", "origin", referencia]).returncode != 0:
        print(f"No se encontró {referencia}; se usa la rama main del curso.")
        subprocess.run(["git", "-C", str(RAIZ), "fetch", "-q", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(RAIZ), "checkout", "-q", "FETCH_HEAD"], check=True)
sys.path.insert(0, str(RAIZ))
print("utils del curso en:", RAIZ)

In [ ]:
# @title Instalar los paquetes que Colab no trae (solo si faltan)
import importlib.util

NECESARIOS = {"pystac-client": "pystac_client", "planetary-computer": "planetary_computer",
              "odc-stac": "odc.stac", "rioxarray": "rioxarray"}
faltan = [paquete for paquete, modulo in NECESARIOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltan], check=True)
print("Instalado:" if faltan else "Ya está todo instalado.", ", ".join(faltan))

In [ ]:
# @title Importar bibliotecas
import time
import warnings
from datetime import date

import folium
import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import plotly.express as px
import pyarrow.parquet as pq
import xarray as xr
from IPython.display import display

# Funciones de apoyo del curso (carpeta utils/)
from utils.extremos import (aislamiento_forest_observaciones, anomalias_observaciones, climatologia_ventana,
                            controlar_calidad_agua, detectar_episodios, solapamiento)
from utils.raster import (convertir_a_reflectancia, convertir_a_temperatura, lonlat_a_crs, mascara_calidad_landsat,
                          mascara_calidad_sentinel2)
from utils.series import (anomalias_zscore, bbox_puntos, cargar_puntos_csv, cargar_ventanas,
                          cargar_ventanas_landsat, climatologia_mensual, construir_series, construir_series_landsat,
                          crs_utm, guardar_series, resumen_validez, tendencia_lineal)
from utils.stac import (agregar_offset, conectar_catalogo, items_legibles, parametros_busqueda, parametros_carga,
                        preparar_items, resumir_imagenes)
from utils.viz import mapa_puntos

warnings.filterwarnings("ignore", category=UserWarning)  # avisos de GDAL
warnings.filterwarnings("ignore", category=RuntimeWarning)  # medianas de ventanas vacías (todo nube): quedan en NaN
pd.set_option("display.width", 200)

# Un color por satélite y uno por punto (paleta Okabe-Ito, apta para daltonismo), los mismos en todo el cuaderno
COLOR_SENSOR = {"Sentinel-2": "#0072B2", "Landsat": "#D55E00"}
PALETA = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000", "#999999", "#F0E442"]
MESES = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]

### Parámetros

* `FECHAS_ESCENA`: dónde buscar el **par de imágenes** del bloque A (una de cada satélite, casi el mismo día). El verano
  tiene menos nubes.
* `FECHAS_S2` y `FECHAS_LANDSAT`: los períodos de las series. Landsat tiene más historia (desde 2013), y eso es lo que
  permite una climatología de varios años en los bloques D y E. Sentinel-2 da más fechas por año.
* `RADIO_M`: radio de la ventana alrededor de cada punto. Es **pequeño a propósito** (45 m): los afluentes miden 30 a
  60 m de ancho y una ventana grande mezclaría agua con ribera. Con 45 m entran unos 20 píxeles de Sentinel-2 y solo unos
  7 de Landsat: ahí se nota la diferencia de resolución.

In [ ]:
FECHAS_ESCENA = ("2025-01-01", "2025-03-31")  # verano: búsqueda del par de imágenes del bloque A
NUBES_ESCENA = 10  # nubes máximas de la escena para el par (%)
FECHAS_S2 = ("2024-01-01", "2025-12-31")  # serie Sentinel-2 (más años = más tiempo de carga)
FECHAS_LANDSAT = ("2013-03-01", "2025-12-31")  # serie Landsat 8/9 (Landsat 8 empieza en marzo de 2013)
NUBES_MAX_SERIES = 70  # filtro de escena laxo para las series: la máscara de nubes de cada píxel hace el resto
RADIO_M = 45  # ventana alrededor de cada punto (m)
FRACCION_MINIMA = 0.5  # una fecha vale si al menos la mitad de la ventana quedó sin nubes
RES_S2_ESCENA = 10  # píxel de la escena Sentinel-2 del bloque A (sube a 20 si Colab se queda sin memoria)
PUNTO_EJEMPLO = "biobio_negrete"  # el punto que se usa en los gráficos de un solo punto
EJECUTAR_EXTREMOS = True  # bloque E (opcional)
RECALCULAR = False  # True: vuelve a descargar las series aunque sus parquet ya existan

SALIDAS = TRABAJO / "salidas"
SALIDAS.mkdir(parents=True, exist_ok=True)
print("Resultados en:", SALIDAS)

### Los puntos

El CSV tiene las columnas del curso (`id, nombre, lat, lon, tipo`) más algunas extra (sección del modelo, río, orden de
aguas arriba a aguas abajo y extremos de la sección). Si el archivo no está, se crea desde el texto de la celda.

In [ ]:
CSV_PUNTOS = TRABAJO / "datos" / "puntos_biobio_rio.csv"
TEXTO_PUNTOS = """id,nombre,lat,lon,tipo,seccion_kml,rio,orden,lat_ini,lon_ini,lat_fin,lon_fin,largo_seccion_m
biobio_alto,Biobío alto (tramo superior del modelo),-37.62013,-72.22818,rio,Perf46,biobio,1,-37.619074,-72.226357,-37.623173,-72.233428,773
biobio_aguas_arriba,Biobío aguas arriba de Duqueco y Bureo,-37.57539,-72.34055,rio,Perf44,biobio,2,-37.575012,-72.33988,-37.579668,-72.348065,889
duqueco,Río Duqueco (tramo final),-37.54274,-72.45334,afluente,Aflu3,duqueco,3,-37.542181,-72.453725,-37.544005,-72.452463,231
bureo,Río Bureo (tramo final),-37.59035,-72.48035,afluente,Aflu1,bureo,4,-37.590286,-72.479731,-37.590529,-72.481981,201
biobio_negrete,Biobío en Negrete (aguas abajo de Duqueco y Bureo),-37.57794,-72.52664,confluencia,Perf30,biobio,5,-37.576407,-72.525641,-37.583559,-72.530295,894
biobio_medio,Biobío entre Negrete y Nacimiento,-37.54647,-72.59077,rio,Perf32,biobio,6,-37.546424,-72.58969,-37.546588,-72.593738,358
biobio_nacimiento,Biobío en Nacimiento (aguas arriba del Vergara),-37.49521,-72.63933,rio,Perf38,biobio,7,-37.497306,-72.640747,-37.490447,-72.636097,865
vergara,Río Vergara en Nacimiento,-37.50267,-72.66567,afluente,Aflu5,vergara,8,-37.502518,-72.666207,-37.503087,-72.664157,192
biobio_bajo_vergara,Biobío aguas abajo del Vergara,-37.46067,-72.68066,confluencia,Perf34,biobio,9,-37.46036,-72.679602,-37.463568,-72.690496,1028
"""
if not CSV_PUNTOS.exists():
    CSV_PUNTOS.parent.mkdir(parents=True, exist_ok=True)
    CSV_PUNTOS.write_text(TEXTO_PUNTOS, encoding="utf-8")
puntos = cargar_puntos_csv(str(CSV_PUNTOS.resolve()))  # ruta completa: el curso no la busca en su carpeta datos/
puntos = puntos.sort_values("orden").reset_index(drop=True)
COLORES = dict(zip(puntos["id"], PALETA))
CRS = crs_utm(puntos["lon"].mean(), puntos["lat"].mean())  # UTM 18 S (EPSG:32718)
BBOX = bbox_puntos(puntos, 2000)  # la ventana de trabajo: los puntos con 2 km de holgura
print("CRS:", CRS, "| bbox [oeste, sur, este, norte]:", [round(v, 3) for v in BBOX])
puntos[["id", "nombre", "lat", "lon", "tipo", "rio", "orden"]]

In [ ]:
mapa = mapa_puntos(puntos, radio_m=RADIO_M)
folium.Rectangle(bounds=[[BBOX[1], BBOX[0]], [BBOX[3], BBOX[2]]], color="red", fill=False, weight=2,
                 tooltip="ventana de trabajo (bloque A)").add_to(mapa)
mapa

### Funciones de este cuaderno

Pocas y a la vista. Los dos satélites llaman distinto a sus bandas; `BANDAS` las traduce a nombres comunes para poder
calcular los mismos índices con los dos.

In [ ]:
BANDAS = {  # nombre común -> banda de cada satélite
    "Sentinel-2": {"azul": "B02", "verde": "B03", "rojo": "B04", "nir": "B08", "swir1": "B11"},
    "Landsat": {"azul": "blue", "verde": "green", "rojo": "red", "nir": "nir08", "swir1": "swir16"},
}


def diferencia_normalizada(a, b):
    """(a - b) / (a + b); 0 / 0 da NaN."""
    return (a - b) / (a + b)


def indices_comunes(refl, sensor):
    """NDWI (verde-NIR), MNDWI (verde-SWIR1), NDVI (NIR-rojo) y reflectancia roja, con los nombres comunes."""
    b = {comun: refl[banda] for comun, banda in BANDAS[sensor].items()}
    return xr.Dataset({"ndwi": diferencia_normalizada(b["verde"], b["nir"]),
                       "mndwi": diferencia_normalizada(b["verde"], b["swir1"]),
                       "ndvi": diferencia_normalizada(b["nir"], b["rojo"]),
                       "rojo": b["rojo"]})


def extension(datos):
    """Límites [oeste, este, sur, norte] en km (UTM) de un raster, para dibujar dos sensores en la misma escala."""
    rx, ry = abs(float(datos.x[1] - datos.x[0])), abs(float(datos.y[1] - datos.y[0]))
    return [(float(datos.x.min()) - rx / 2) / 1000, (float(datos.x.max()) + rx / 2) / 1000,
            (float(datos.y.min()) - ry / 2) / 1000, (float(datos.y.max()) + ry / 2) / 1000]


def lado_a_lado(capa_s2, capa_l, titulo, cmap=None, vmin=None, vmax=None, etiqueta=None, zoom=None, figsize=(15, 5.5)):
    """La misma variable de los dos satélites, una al lado de la otra, en coordenadas UTM (km) y con la misma escala."""
    fig, ejes = plt.subplots(1, 2, figsize=figsize, sharex=True, sharey=True)
    for eje, capa, nombre in zip(ejes, [capa_s2, capa_l], ["Sentinel-2", "Landsat"]):
        datos = capa if isinstance(capa, xr.DataArray) else capa[0]
        imagen = capa if isinstance(capa, xr.DataArray) else capa[1]
        dibujo = eje.imshow(np.asarray(imagen), extent=extension(datos), cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
        eje.set_title(f"{nombre}: {titulo}", fontsize=10)
        eje.set_xlabel("este (km, UTM 18 S)")
        for p in puntos.itertuples():
            x, y = lonlat_a_crs(p.lon, p.lat, CRS)
            eje.plot(x / 1000, y / 1000, "^", color="k", mec="white", ms=6)
        if etiqueta and np.asarray(imagen).ndim == 2:
            fig.colorbar(dibujo, ax=eje, fraction=0.03, label=etiqueta)
    ejes[0].set_ylabel("norte (km)")
    if zoom:
        ejes[0].set_xlim(zoom[0], zoom[1])
        ejes[0].set_ylim(zoom[2], zoom[3])
    plt.tight_layout()
    plt.show()

## Bloque A. Una imagen de cada satélite (basado en el bloque 03)

### A.1 Buscar un par de imágenes casi del mismo día

Buscamos las dos colecciones sobre la ventana de trabajo con la llamada nativa `catalogo.search(...)`. Los dos satélites
pasan sobre Chile central cerca de las 14:40 UTC, así que hay días en que pasan **los dos**: esos pares son la mejor forma
de comparar, porque el río y la atmósfera son los mismos.

In [ ]:
catalogo = conectar_catalogo()  # Client.open(...) de pystac_client, con la firma de Planetary Computer
items_s2_escena = preparar_items(catalogo.search(**parametros_busqueda(
    "sentinel-2", BBOX, FECHAS_ESCENA, nubes_max=NUBES_ESCENA, catalogo=catalogo)).item_collection())
items_l_escena = list(catalogo.search(**parametros_busqueda(
    "landsat", BBOX, FECHAS_ESCENA, nubes_max=NUBES_ESCENA, catalogo=catalogo)).items())
print(f"Sentinel-2: {len(items_s2_escena)} imágenes | Landsat: {len(items_l_escena)} imágenes")
display(resumir_imagenes(items_s2_escena).head())  # fecha, nubes, baseline y offset de cada imagen Sentinel-2 (bloque 2)


def por_dia(items):
    """Nubes medias por día (un día puede tener varias escenas: tiles o filas vecinas)."""
    tabla = pd.DataFrame({"dia": [i.datetime.date() for i in items], "nubes": [i.properties.get("eo:cloud_cover") for i in items]})
    return tabla.groupby("dia")["nubes"].mean()


dias_s2, dias_l = por_dia(items_s2_escena), por_dia(items_l_escena)
pares = []
for dia_l, nubes_l in dias_l.items():
    distancia = pd.Series([abs((d - dia_l).days) for d in dias_s2.index], index=dias_s2.index)
    dia_s2 = distancia.idxmin()
    pares.append({"dia_landsat": dia_l, "dia_sentinel2": dia_s2, "dias_entre": int(distancia.min()),
                  "nubes_landsat": round(nubes_l, 1), "nubes_sentinel2": round(dias_s2[dia_s2], 1)})
pares = pd.DataFrame(pares).sort_values(["dias_entre", "nubes_landsat", "nubes_sentinel2"]).reset_index(drop=True)
display(pares.head(8))
elegido = pares.iloc[0]
print(f"Par elegido: Landsat {elegido.dia_landsat} y Sentinel-2 {elegido.dia_sentinel2} ({elegido.dias_entre} días entre ambas)")

### A.2 Cargar las dos escenas y pasar de números digitales a reflectancia

Mismo patrón para los dos: `parametros_carga` arma los argumentos y `odc.stac.load` lee **solo la ventana de trabajo**
(gracias al formato COG, bloque 03). Lo que cambia es la calibración:

* **Sentinel-2:** `reflectancia = DN × 0,0001 + offset`, con el offset de **esa imagen** según su baseline (0 o -0,1;
  `agregar_offset` lo deja en una coordenada y `convertir_a_reflectancia` lo usa).
* **Landsat:** `reflectancia = DN × 0,0000275 - 0,2` y `temperatura (K) = DN × 0,00341802 + 149` (banda térmica `lwir11`).

In [ ]:
items_s2_dia = [i for i in items_s2_escena if i.datetime.date() == elegido.dia_sentinel2]
items_l_dia = [i for i in items_l_escena if i.datetime.date() == elegido.dia_landsat]

t0 = time.time()
carga_s2 = parametros_carga("sentinel-2", ["B02", "B03", "B04", "B08", "B11", "SCL"], RES_S2_ESCENA, CRS, catalogo, BBOX,
                            chunks={"x": 2048, "y": 2048})
escena_s2 = agregar_offset(odc.stac.load(items_s2_dia, **carga_s2), items_s2_dia).isel(time=0).compute()
carga_l = parametros_carga("landsat", ["blue", "green", "red", "nir08", "swir16", "lwir11", "qa_pixel"], 30, CRS, catalogo, BBOX,
                           chunks={"x": 2048, "y": 2048})
escena_l = odc.stac.load(items_l_dia, **carga_l).isel(time=0).compute()
print(f"Lectura: {time.time() - t0:.0f} s")
print(f"Sentinel-2: {escena_s2.sizes['x']} x {escena_s2.sizes['y']} píxeles de {RES_S2_ESCENA} m "
      f"(offset {float(escena_s2['offset_reflectancia']):+.1f})")
print(f"Landsat:    {escena_l.sizes['x']} x {escena_l.sizes['y']} píxeles de 30 m")
print(f"La misma zona tiene {escena_s2.sizes['x'] * escena_s2.sizes['y'] / (escena_l.sizes['x'] * escena_l.sizes['y']):.0f} "
      "veces más píxeles en Sentinel-2")

In [ ]:
refl_s2 = convertir_a_reflectancia(escena_s2[list(BANDAS["Sentinel-2"].values())], "sentinel-2").astype("float32")
valido_s2 = mascara_calidad_sentinel2(escena_s2["SCL"]) & (escena_s2["SCL"] != 0)
refl_l = convertir_a_reflectancia(escena_l[list(BANDAS["Landsat"].values())], "landsat").astype("float32")
valido_l = mascara_calidad_landsat(escena_l["qa_pixel"].fillna(1))
temperatura_l = convertir_a_temperatura(escena_l["lwir11"]).where(valido_l)

ind_s2 = indices_comunes(refl_s2, "Sentinel-2").where(valido_s2)
ind_l = indices_comunes(refl_l, "Landsat").where(valido_l)
print(f"Píxeles válidos (sin nubes): Sentinel-2 {float(valido_s2.mean()):.0%} | Landsat {float(valido_l.mean()):.0%}")

### A.3 Color verdadero y falso color

Las reflectancias negativas (píxeles muy oscuros) quedaron en NaN: para **mirar** se pintan de negro, para **calcular**
se dejan en NaN (bloque 03).

In [ ]:
def rgb(refl, sensor, bandas=("rojo", "verde", "azul"), brillo=3.5):
    return np.clip(np.dstack([refl[BANDAS[sensor][b]].fillna(0).values for b in bandas]) * brillo, 0, 1)


lado_a_lado((refl_s2, rgb(refl_s2, "Sentinel-2")), (refl_l, rgb(refl_l, "Landsat")), "color verdadero")
lado_a_lado((refl_s2, rgb(refl_s2, "Sentinel-2", ("nir", "rojo", "verde"), 3.0)),
            (refl_l, rgb(refl_l, "Landsat", ("nir", "rojo", "verde"), 3.0)), "falso color (NIR, rojo, verde)")

### A.4 Índices: agua, vegetación y turbidez

| Variable | Fórmula | Qué muestra |
|---|---|---|
| NDWI (McFeeters) | (verde - NIR) / (verde + NIR) | agua abierta (positivo) |
| MNDWI (Xu) | (verde - SWIR1) / (verde + SWIR1) | agua, mejor con sedimento; SWIR1 es de 20 m en Sentinel-2 |
| NDVI | (NIR - rojo) / (NIR + rojo) | vegetación |
| Reflectancia roja sobre el agua | banda roja | turbidez (sedimento en suspensión) |

In [ ]:
lado_a_lado(ind_s2["ndwi"], ind_l["ndwi"], "NDWI", cmap="RdBu", vmin=-0.8, vmax=0.8, etiqueta="NDWI")
lado_a_lado(ind_s2["ndvi"], ind_l["ndvi"], "NDVI", cmap="RdYlGn", vmin=-0.2, vmax=0.9, etiqueta="NDVI")
lado_a_lado(ind_s2["rojo"].where(ind_s2["mndwi"] > 0), ind_l["rojo"].where(ind_l["mndwi"] > 0),
            "reflectancia roja sobre el agua (turbidez)", cmap="YlOrBr", vmin=0, vmax=0.12, etiqueta="reflectancia")

### A.5 Lo que solo Landsat tiene: la temperatura de superficie

La banda térmica (100 m nativos) mide la temperatura de la **piel** de la superficie. En verano el río es mucho más frío
que los campos y las barras secas que lo rodean. Sentinel-2 no tiene banda térmica.

In [ ]:
fig, eje = plt.subplots(figsize=(13, 5.5))
dibujo = eje.imshow(temperatura_l, extent=extension(escena_l), cmap="RdYlBu_r",
                    vmin=float(temperatura_l.quantile(0.02)), vmax=float(temperatura_l.quantile(0.98)))
for p in puntos.itertuples():
    x, y = lonlat_a_crs(p.lon, p.lat, CRS)
    eje.plot(x / 1000, y / 1000, "^", color="k", mec="white", ms=6)
fig.colorbar(dibujo, ax=eje, fraction=0.03, label="temperatura de superficie (°C)")
eje.set_title(f"Landsat {elegido.dia_landsat}: temperatura de superficie")
eje.set_xlabel("este (km, UTM 18 S)")
eje.set_ylabel("norte (km)")
plt.tight_layout()
plt.show()

agua_l = ind_l["mndwi"] > 0
print(f"Temperatura mediana: agua {float(temperatura_l.where(agua_l).median()):.1f} °C | "
      f"tierra {float(temperatura_l.where(~agua_l).median()):.1f} °C")

### A.6 Zoom: 10 m frente a 30 m en una confluencia

La confluencia del Vergara en Nacimiento, en un cuadrado de 3 km. Aquí se ve por qué un afluente de 50 m de ancho se
distingue en Sentinel-2 (5 píxeles) y en Landsat queda en uno o dos píxeles **mixtos** (agua y orilla a la vez).

In [ ]:
vergara = puntos.set_index("id").loc["vergara"]
xv, yv = lonlat_a_crs(vergara.lon, vergara.lat, CRS)
zoom = [(xv - 1500) / 1000, (xv + 1500) / 1000, (yv - 1500) / 1000, (yv + 1500) / 1000]
lado_a_lado((refl_s2, rgb(refl_s2, "Sentinel-2")), (refl_l, rgb(refl_l, "Landsat")), "color verdadero, confluencia del Vergara",
            zoom=zoom, figsize=(13, 6.5))
lado_a_lado(ind_s2["mndwi"], ind_l["mndwi"], "MNDWI, confluencia del Vergara", cmap="RdBu", vmin=-0.8, vmax=0.8,
            etiqueta="MNDWI", zoom=zoom, figsize=(13, 6))

### A.7 Cuánta agua ve cada satélite

El área con MNDWI > 0 en toda la ventana, y el histograma del NDWI: si los dos satélites ven lo mismo, las curvas se
parecen; si Landsat tiene más píxeles mixtos, su histograma se "rellena" entre los dos grupos (agua y tierra).

In [ ]:
area_pixel = {"Sentinel-2": RES_S2_ESCENA ** 2, "Landsat": 30 ** 2}
filas = []
for sensor, ind in (("Sentinel-2", ind_s2), ("Landsat", ind_l)):
    agua = ind["mndwi"] > 0
    filas.append({"satélite": sensor, "área de agua (km²)": float(agua.sum()) * area_pixel[sensor] / 1e6,
                  "% de los píxeles válidos": 100 * float(agua.sum() / ind["mndwi"].notnull().sum())})
display(pd.DataFrame(filas).round(2))

fig, eje = plt.subplots(figsize=(9, 3.6))
for sensor, ind in (("Sentinel-2", ind_s2), ("Landsat", ind_l)):
    valores = ind["ndwi"].values.ravel()
    eje.hist(valores[np.isfinite(valores)], bins=120, range=(-1, 1), density=True, histtype="step", lw=2,
             color=COLOR_SENSOR[sensor], label=sensor)
eje.axvline(0, color="k", ls="--", lw=0.8)
eje.set_xlabel("NDWI")
eje.set_ylabel("densidad")
eje.set_title("Distribución del NDWI en la ventana de trabajo")
eje.legend()
eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### A.8 El valor del píxel en cada punto

El píxel más cercano a cada punto con cada satélite (`.sel(..., method="nearest")`). En el Biobío (cientos de metros de
ancho) los dos deberían coincidir; en los afluentes, el píxel de Landsat mezcla agua y orilla y su NDWI baja.

In [ ]:
filas = []
for p in puntos.itertuples():
    x, y = lonlat_a_crs(p.lon, p.lat, CRS)
    s2 = ind_s2.sel(x=x, y=y, method="nearest")
    ls = ind_l.sel(x=x, y=y, method="nearest")
    filas.append({"punto": p.id, "tipo": p.tipo, "NDWI S2": float(s2["ndwi"]), "NDWI L": float(ls["ndwi"]),
                  "rojo S2": float(s2["rojo"]), "rojo L": float(ls["rojo"]),
                  "temperatura L (°C)": float(temperatura_l.sel(x=x, y=y, method="nearest"))})
display(pd.DataFrame(filas).set_index("punto").round(3))
del escena_s2, refl_s2  # libera memoria antes de las series

## Bloque B. Series temporales con los dos satélites (basado en el bloque 04)

De una imagen a una serie: el valor de cada punto fecha tras fecha. Igual que en el bloque 04 se cargan **ventanas
pequeñas** alrededor de cada punto (no la ventana completa), se descartan los píxeles con nubes con la máscara de cada
satélite y se resume cada fecha con la mediana de la ventana. Lo hacen las funciones del curso:

| | Sentinel-2 | Landsat |
|---|---|---|
| Carga | `cargar_ventanas` | `cargar_ventanas_landsat` |
| Máscara, calibración, índices y resumen | `construir_series` (SCL) | `construir_series_landsat` (qa_pixel y qa_radsat) |
| Variables | `ndwi`, `ndci`, `b04` (rojo), `ndvi` | `st_c` (temperatura), `verde`, `rojo`, `nir`, `swir1`, `ndwi`, `ndvi` |

Se trabaja **año por año** y cada año se guarda en un parquet: si Colab se corta, al volver a ejecutar no se descarga de
nuevo lo que ya estaba. Landsat de 2013 a 2025 tarda bastante (del orden de media hora o más, según la red).

In [ ]:
def serie_por_anios(sensor, fechas):
    """Busca, carga y resume un satélite año por año; cada año queda en salidas/serie_<sensor>_<año>.parquet."""
    clave = "s2" if sensor == "sentinel-2" else "landsat"
    tablas = []
    for anio in range(int(fechas[0][:4]), int(fechas[1][:4]) + 1):
        ruta = SALIDAS / f"serie_{clave}_{anio}.parquet"
        if ruta.exists() and not RECALCULAR:
            tablas.append(pd.read_parquet(ruta))
            print(f"{sensor} {anio}: leído de {ruta.name}")
            continue
        rango = (max(fechas[0], f"{anio}-01-01"), min(fechas[1], f"{anio}-12-31"))
        busqueda = catalogo.search(**parametros_busqueda(sensor, bbox_puntos(puntos, RADIO_M), rango,
                                                         nubes_max=NUBES_MAX_SERIES, catalogo=catalogo))
        t0 = time.time()
        if sensor == "sentinel-2":
            items = preparar_items(busqueda.item_collection(), verbose=False)  # sin versiones repetidas
            items, danados = items_legibles(items, ["B03", "B04", "B05", "B08", "SCL"])
            ventanas = cargar_ventanas(items, puntos, radio_m=RADIO_M, crs=CRS, catalogo=catalogo, verbose=False)
            tabla = construir_series(ventanas, puntos, radio_m=RADIO_M, fraccion_minima=FRACCION_MINIMA)
        else:
            items = list(busqueda.items())
            items, danados = items_legibles(items, ["green", "red", "nir08", "swir16", "lwir11", "qa_pixel", "qa_radsat"])
            ventanas = cargar_ventanas_landsat(items, puntos, radio_m=RADIO_M, crs=CRS, verbose=False)
            tabla = construir_series_landsat(ventanas, puntos, radio_m=RADIO_M, fraccion_minima=FRACCION_MINIMA)
        del ventanas
        tabla.to_parquet(ruta, index=False)
        tablas.append(tabla)
        print(f"{sensor} {anio}: {len(items)} imágenes ({len(danados)} con archivos dañados), "
              f"{tabla['fecha'].nunique()} fechas, {time.time() - t0:.0f} s")
    serie = pd.concat(tablas, ignore_index=True).sort_values(["id", "fecha"]).reset_index(drop=True)
    return serie.merge(puntos[["id", "rio", "orden"]], on="id")


serie_s2 = serie_por_anios("sentinel-2", FECHAS_S2)
serie_l = serie_por_anios("landsat", FECHAS_LANDSAT)
print(f"\nSentinel-2: {len(serie_s2)} filas | Landsat: {len(serie_l)} filas")

### B.1 ¿Cuántas fechas sirven con cada satélite?

`resumen_validez` del curso: fechas con cobertura y fechas con un valor útil (no tapadas por nubes). La segunda tabla
cuenta las **observaciones válidas por año**: la diferencia de revisita entre los dos satélites.

In [ ]:
validez = pd.concat({"Sentinel-2": resumen_validez(serie_s2, "ndwi")[["fechas", "fechas_validas", "porcentaje_valido"]],
                     "Landsat": resumen_validez(serie_l, "ndwi")[["fechas", "fechas_validas", "porcentaje_valido"]]}, axis=1)
display(validez)

por_anio = pd.DataFrame({
    "Sentinel-2": serie_s2.dropna(subset=["ndwi"]).groupby(serie_s2["fecha"].dt.year).size() / puntos.shape[0],
    "Landsat": serie_l.dropna(subset=["ndwi"]).groupby(serie_l["fecha"].dt.year).size() / puntos.shape[0]})
fig, eje = plt.subplots(figsize=(10, 3.5))
por_anio.plot.bar(ax=eje, color=[COLOR_SENSOR["Sentinel-2"], COLOR_SENSOR["Landsat"]], width=0.8)
eje.set_ylabel("observaciones válidas por punto")
eje.set_xlabel("")
eje.set_title("Observaciones sin nubes por año (promedio de los 9 puntos)")
eje.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

### B.2 Guardar las series en parquet con metadatos (FAIR)

Como en el bloque 04: el archivo lleva dentro su título, fuente, licencia, parámetros y el significado de cada columna.

In [ ]:
comunes = {"generado": date.today().isoformat(), "radio_m": str(RADIO_M), "fraccion_minima": str(FRACCION_MINIMA),
           "codigo": f"comparacion_sentinel2_landsat; utils del curso geomatica-aplicada, commit {COMMIT_CURSO}",
           "puntos": "thalweg de 9 secciones del Biobío medio y sus afluentes Vergara, Duqueco y Bureo"}
guardar_series(serie_s2, SALIDAS / "series_sentinel2_biobio.parquet", {
    **comunes, "titulo": f"Series Sentinel-2 L2A en el Biobío medio ({FECHAS_S2[0]} a {FECHAS_S2[1]})",
    "fuente": "ESA Copernicus Sentinel-2 L2A, Microsoft Planetary Computer (sentinel-2-l2a)",
    "licencia": "Datos Copernicus Sentinel de acceso libre. Contiene datos Copernicus Sentinel modificados.",
    "columnas": {"ndwi": "(B03 - B08) / (B03 + B08)", "ndci": "(B05 - B04) / (B05 + B04)", "b04": "reflectancia roja",
                 "ndvi": "(B08 - B04) / (B08 + B04)", "pixeles_validos": "fracción de la ventana sin nubes (SCL)"}})
guardar_series(serie_l, SALIDAS / "series_landsat_biobio.parquet", {
    **comunes, "titulo": f"Series Landsat 8/9 C2 L2 en el Biobío medio ({FECHAS_LANDSAT[0]} a {FECHAS_LANDSAT[1]})",
    "fuente": "USGS Landsat 8 y 9 Collection 2 Level-2, Microsoft Planetary Computer (landsat-c2-l2)",
    "licencia": "Dominio público (USGS)",
    "columnas": {"st_c": "temperatura de superficie, °C (DN x 0,00341802 + 149 - 273,15)", "rojo": "reflectancia roja",
                 "ndwi": "(verde - NIR) / (verde + NIR)", "ndvi": "(NIR - rojo) / (NIR + rojo)",
                 "pixeles_validos": "fracción de la ventana sin nubes (qa_pixel y qa_radsat)"}})
print("Metadatos:", [k.decode() for k in pq.read_schema(SALIDAS / "series_landsat_biobio.parquet").metadata if k != b"pandas"])

### B.3 Las dos series juntas

Un punto, tres variables comparables (NDWI, NDVI y reflectancia roja). Los marcadores son las observaciones: **no se
rellenan los vacíos** (bloque 06). Gráfico interactivo: pasa el mouse, haz zoom, oculta un satélite en la leyenda.

In [ ]:
COMPARABLES = {"ndwi": ("ndwi", "ndwi"), "ndvi": ("ndvi", "ndvi"), "rojo (turbidez)": ("b04", "rojo")}  # nombre: (S2, Landsat)


def largo_dos_sensores(id_punto):
    partes = []
    for nombre, (col_s2, col_l) in COMPARABLES.items():
        for sensor, tabla, col in (("Sentinel-2", serie_s2, col_s2), ("Landsat", serie_l, col_l)):
            sub = tabla[tabla["id"] == id_punto][["fecha", col]].dropna().rename(columns={col: "valor"})
            partes.append(sub.assign(variable=nombre, satelite=sensor))
    return pd.concat(partes, ignore_index=True)


largo = largo_dos_sensores(PUNTO_EJEMPLO)
fig = px.scatter(largo, x="fecha", y="valor", color="satelite", facet_row="variable", height=750,
                 color_discrete_map=COLOR_SENSOR, title=f"{PUNTO_EJEMPLO}: Sentinel-2 y Landsat",
                 labels={"fecha": "Fecha", "valor": "", "satelite": "Satélite"})
fig.update_yaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.write_html(str(SALIDAS / f"series_{PUNTO_EJEMPLO}.html"), include_plotlyjs="cdn")
fig.show()

## Bloque C. Comparación directa: el mismo día con los dos satélites

Cuando los dos pasan **el mismo día** sobre el mismo punto, sus diferencias vienen del sensor (resolución, bandas,
corrección atmosférica) y no del río. Juntamos esas fechas y medimos:

* **sesgo** = media de (Landsat - Sentinel-2);
* **RMSE** = raíz del error cuadrático medio;
* **r** = correlación de Pearson.

In [ ]:
dia_s2 = serie_s2.assign(dia=serie_s2["fecha"].dt.normalize())
dia_l = serie_l.assign(dia=serie_l["fecha"].dt.normalize())
mismo_dia = dia_s2.merge(dia_l, on=["id", "dia"], suffixes=("_s2", "_l"))
print(f"{len(mismo_dia)} pares punto-día con los dos satélites ({mismo_dia['dia'].nunique()} días distintos)")

filas = []
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.6))
for eje, (nombre, (col_s2, col_l)) in zip(ejes, COMPARABLES.items()):
    a = f"{col_s2}_s2" if col_s2 == col_l else col_s2
    b = f"{col_l}_l" if col_s2 == col_l else col_l
    par = mismo_dia[["id", "tipo_s2", a, b]].dropna()
    if par.empty:
        continue
    diferencia = par[b] - par[a]
    filas.append({"variable": nombre, "pares": len(par), "sesgo (L - S2)": diferencia.mean(),
                  "RMSE": np.sqrt((diferencia ** 2).mean()), "r": par[[a, b]].corr().iloc[0, 1]})
    for tipo, marcador in (("rio", "o"), ("confluencia", "s"), ("afluente", "^")):
        sub = par[par["tipo_s2"] == tipo]
        eje.plot(sub[a], sub[b], marcador, ms=5, alpha=0.6, label=tipo)
    limites = [min(par[a].min(), par[b].min()), max(par[a].max(), par[b].max())]
    eje.plot(limites, limites, "k--", lw=1, label="1:1")
    eje.set_xlabel(f"Sentinel-2 ({col_s2})")
    eje.set_ylabel(f"Landsat ({col_l})")
    eje.set_title(nombre)
    eje.grid(alpha=0.3)
ejes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()
display(pd.DataFrame(filas).set_index("variable").round(3))

Y por tipo de punto: ¿la diferencia de NDWI es mayor en los afluentes angostos (píxel mixto de Landsat)?

In [ ]:
mismo_dia["dif_ndwi"] = mismo_dia["ndwi_l"] - mismo_dia["ndwi_s2"]
display(mismo_dia.groupby(["tipo_s2", "id"])["dif_ndwi"].agg(pares="count", mediana="median").round(3))

**Cómo leerlo.** En el cauce principal (cientos de metros de ancho) los dos satélites deberían dar casi lo mismo
(r alto, sesgo chico): sirven para lo mismo y se pueden **combinar** para tener más fechas. En los afluentes, el NDWI de
Landsat queda por debajo: con píxeles de 30 m la ventana mezcla agua con ribera. Para ríos angostos, Sentinel-2.

## Bloque D. Exploración, climatología y anomalías (basado en el bloque 05)

### D.1 Estadísticos por punto y satélite

Media, mediana y percentiles de la reflectancia roja (turbidez) con cada satélite. La mediana resiste mejor los valores
extremos (una nube que se coló, una crecida).

In [ ]:
def resumen_variable(tabla, variable):
    grupo = tabla.groupby("id", sort=False)[variable]
    return pd.DataFrame({"n": grupo.count(), "media": grupo.mean(), "mediana": grupo.median(), "p10": grupo.quantile(0.1),
                         "p90": grupo.quantile(0.9), "max": grupo.max()}).reindex(puntos["id"])


display(pd.concat({"Sentinel-2 (b04)": resumen_variable(serie_s2, "b04"),
                   "Landsat (rojo)": resumen_variable(serie_l, "rojo")}, axis=1).round(3))

fig, eje = plt.subplots(figsize=(13, 4))
posiciones = np.arange(len(puntos))
for desplazamiento, (sensor, tabla, col) in zip((-0.2, 0.2), (("Sentinel-2", serie_s2, "b04"), ("Landsat", serie_l, "rojo"))):
    datos = [tabla.loc[tabla["id"] == p, col].dropna() for p in puntos["id"]]
    caja = eje.boxplot(datos, positions=posiciones + desplazamiento, widths=0.35, patch_artist=True, showfliers=False)
    for parche in caja["boxes"]:
        parche.set_facecolor(COLOR_SENSOR[sensor])
        parche.set_alpha(0.6)
    eje.plot([], [], "s", color=COLOR_SENSOR[sensor], label=sensor)
eje.set_xticks(posiciones, labels=[p.replace("_", "\n") for p in puntos["id"]], fontsize=8)
eje.set_ylabel("reflectancia roja")
eje.set_title("Turbidez por punto, de aguas arriba (izquierda) a aguas abajo")
eje.legend()
eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### D.2 Fechas irregulares y media móvil en días

Las dos series son irregulares (nubes), pero Landsat lo es más. Con fechas irregulares conviene una ventana en **días**
y un mínimo de observaciones (`min_periods`): 30 días alcanzan para Sentinel-2; Landsat necesita 60.

In [ ]:
fig, eje = plt.subplots(figsize=(12, 4))
for sensor, tabla, col, ventana in (("Sentinel-2", serie_s2, "b04", "30D"), ("Landsat", serie_l, "rojo", "60D")):
    s = tabla[tabla["id"] == PUNTO_EJEMPLO].set_index("fecha")[col].dropna()
    s = s[s.index >= pd.Timestamp(FECHAS_S2[0])]  # el mismo período para los dos
    eje.plot(s.index, s, "o", ms=3, alpha=0.4, color=COLOR_SENSOR[sensor])
    eje.plot(s.index, s.rolling(ventana, min_periods=3, center=True).mean(), lw=2, color=COLOR_SENSOR[sensor],
             label=f"{sensor}: media móvil de {ventana[:-1]} días")
eje.set_ylabel("reflectancia roja")
eje.set_title(f"{PUNTO_EJEMPLO}: observaciones (puntos) y media móvil (línea)")
eje.legend(fontsize=8)
eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### D.3 Ciclo anual

Mediana mensual por punto, juntando todos los años. La serie larga de Landsat también da el ciclo de la **temperatura
del río**, que Sentinel-2 no puede medir.

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(16, 4.3))
for eje, (titulo, tabla, col, cmap) in zip(ejes, (("Sentinel-2: reflectancia roja", serie_s2, "b04", "YlOrBr"),
                                                   ("Landsat: reflectancia roja", serie_l, "rojo", "YlOrBr"),
                                                   ("Landsat: temperatura (°C)", serie_l, "st_c", "RdYlBu_r"))):
    mensual = tabla.groupby(["id", tabla["fecha"].dt.month])[col].median().unstack().reindex(index=puntos["id"], columns=range(1, 13))
    dibujo = eje.imshow(mensual.values, aspect="auto", cmap=cmap)
    eje.set_yticks(range(len(mensual)), labels=mensual.index, fontsize=7)
    eje.set_xticks(range(12), labels=MESES, fontsize=7)
    eje.set_title(titulo, fontsize=10)
    plt.colorbar(dibujo, ax=eje, fraction=0.04)
plt.tight_layout()
plt.show()

### D.4 Tendencia: lo que permite una serie larga

El bloque 05 advierte que con **un año** una pendiente mezcla la estación con el cambio. Con Landsat (más de 10 años)
se puede quitar primero el ciclo anual (anomalía respecto de la media de cada mes) y ajustar la recta a las anomalías.
El valor p supone observaciones independientes y por eso **sobreestima** la significancia: úsalo con cuidado.

In [ ]:
BASE_LANDSAT = (int(serie_l["fecha"].dt.year.min()), int(serie_l["fecha"].dt.year.max()))
filas = []
for p in puntos["id"]:
    for col in ("st_c", "rojo", "ndvi"):
        s = serie_l[serie_l["id"] == p].set_index("fecha")[col].dropna()
        if len(s) < 10:
            continue
        clima = climatologia_mensual(s)
        anomalia = s - s.index.month.map(clima["media"]).to_numpy()
        t = tendencia_lineal(anomalia)
        filas.append({"punto": p, "variable": col, "n": t.n, "pendiente por década": 10 * t.pendiente_anual, "p": t.p_valor})
tendencias = pd.DataFrame(filas)
display(tendencias.pivot(index="punto", columns="variable", values="pendiente por década").reindex(puntos["id"]).round(3))
print(f"Landsat {BASE_LANDSAT[0]}-{BASE_LANDSAT[1]}: pendiente de la anomalía, en unidades de la variable por década "
      "(st_c en °C por década).")

### D.5 Anomalías (z-score)

`z = (valor - media) / desviación`. Con la serie corta de Sentinel-2 se usa la referencia de **toda la serie**
(`referencia="total"`); con la larga de Landsat se puede usar la del **mismo mes** de los años base (`referencia="mes"`),
que quita la estación. Un |z| > 2 es poco habitual. El gráfico muestra la anomalía mensual media de la temperatura del
río: **un mes sin barra no es anomalía cero, es un mes sin dato**.

In [ ]:
filas = []
for p in puntos["id"]:
    s2 = serie_s2[serie_s2["id"] == p].set_index("fecha")["b04"].dropna()
    ls = serie_l[serie_l["id"] == p].set_index("fecha")["rojo"].dropna()
    z_s2 = anomalias_zscore(s2, referencia="total")
    z_l = anomalias_zscore(ls, referencia="mes", base=BASE_LANDSAT)
    filas.append({"punto": p, "S2: fechas |z| > 2": int((z_s2.abs() > 2).sum()), "S2: n": len(s2),
                  "Landsat: fechas |z| > 2": int((z_l.abs() > 2).sum()), "Landsat: n": len(ls)})
display(pd.DataFrame(filas).set_index("punto"))

st = serie_l[serie_l["id"] == PUNTO_EJEMPLO].set_index("fecha")["st_c"].dropna()
z_mensual = anomalias_zscore(st, referencia="mes", base=BASE_LANDSAT).resample("MS").mean()
fig, eje = plt.subplots(figsize=(13, 3.6))
eje.bar(z_mensual.index, z_mensual.fillna(0), width=25, color=np.where(z_mensual.fillna(0) >= 0, "#D55E00", "#0072B2"))
eje.axhline(0, color="k", lw=0.7)
eje.set_ylabel("anomalía mensual (z)")
eje.set_title(f"{PUNTO_EJEMPLO}: anomalía de la temperatura del río respecto del mismo mes ({BASE_LANDSAT[0]}-{BASE_LANDSAT[1]})")
eje.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Bloque E. Eventos extremos (opcional, basado en el bloque 07)

Una **anomalía** es un valor raro; un **episodio** es una anomalía que **persiste**. El método del bloque 07 (idea de
Hobday, adaptada a series irregulares):

1. **Control de calidad**: la reflectancia delata las nubes que la máscara no vio.
2. **Climatología por ventana de días**: para cada día del año, media y percentiles 10 y 90 de las observaciones de
   todos los años a ±15 días. Con la serie corta de Sentinel-2 cada año se compara con **los otros años** (ventana de
   ±30 días para juntar suficientes observaciones), para que un evento fuerte no infle su propia referencia.
3. **Episodio**: al menos 2 observaciones seguidas sobre el p90 (o bajo el p10), sin huecos de más de N días
   (24 en Landsat, 12 en Sentinel-2, por su revisita).
4. **Isolation Forest**: aprendizaje automático que marca combinaciones raras de varias variables a la vez.

In [ ]:
if EJECUTAR_EXTREMOS:
    # 1. Control de calidad. El curso usa rojo < 0,05 y SWIR1 < 0,02 para agua de mar clara. Un río turbio tiene más
    #    reflectancia roja, así que se relaja el rojo; el SWIR1 sigue delatando nubes (y píxeles de orilla).
    tipos_agua = tuple(puntos["tipo"].unique())
    serie_l_qc = controlar_calidad_agua(serie_l, rojo_max=0.30, swir1_max=0.05, tipos=tipos_agua)
    descartadas = (serie_l["st_c"].notna() & serie_l_qc["st_c"].isna()).groupby(serie_l["id"]).sum()
    print("Observaciones Landsat descartadas por sospecha de nube u orilla:")
    print(descartadas.reindex(puntos["id"]).to_string())

### E.1 Episodios de temperatura del río (solo Landsat)

In [ ]:
def episodios(tabla, variable, base, brecha, tipos=("calor", "frio"), ventana=31, minimo_obs=8, dejar_fuera_anio=False):
    """
    Climatología por ventana y episodios de cada punto (como episodios_por_punto del curso, con base y brecha elegibles).

    Con `dejar_fuera_anio=True` cada año se compara con la climatología de **los otros años**: con una serie corta
    (2 o 3 años de Sentinel-2) un evento fuerte inflaría su propia referencia y no se detectaría.
    """
    partes = []
    for p in puntos["id"]:
        s = tabla[tabla["id"] == p].set_index("fecha")[variable]
        tramos = [(None, s)] if not dejar_fuera_anio else [(a, s[s.index.year == a]) for a in sorted(set(s.index.year))]
        for anio, tramo in tramos:
            referencia = s if anio is None else s[s.index.year != anio]
            try:
                clima = climatologia_ventana(referencia, base=base, ventana=ventana, minimo_obs=minimo_obs)
            except ValueError:
                continue
            for tipo in tipos:
                ev = detectar_episodios(tramo, clima, tipo, min_obs=2, brecha_max_dias=brecha)
                if not ev.empty:
                    partes.append(ev.assign(punto=p, variable=variable))
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(
        columns=["punto", "variable", "tipo", "inicio", "fin", "duracion", "n_obs", "intensidad_max"])


if EJECUTAR_EXTREMOS:
    base_l = (BASE_LANDSAT[0], min(BASE_LANDSAT[1], 2024))
    ep_temp = episodios(serie_l_qc, "st_c", base_l, brecha=24)
    print(f"{len(ep_temp)} episodios de temperatura (base {base_l[0]}-{base_l[1]})")
    display(ep_temp.groupby(["punto", "tipo"]).agg(episodios=("inicio", "size"), obs_max=("n_obs", "max"),
                                                   intensidad_max=("intensidad_max", lambda x: x.loc[x.abs().idxmax()])).round(2))

    s = serie_l_qc[serie_l_qc["id"] == PUNTO_EJEMPLO].set_index("fecha")["st_c"]
    clima = climatologia_ventana(s, base=base_l)
    an = anomalias_observaciones(s, clima).dropna(subset=["valor"])
    fig, eje = plt.subplots(figsize=(13, 4))
    eje.plot(an.index, an["anomalia"], "o", ms=3, color="k")
    for ev in ep_temp[ep_temp["punto"] == PUNTO_EJEMPLO].itertuples():
        eje.axvspan(ev.inicio, ev.fin + pd.Timedelta(days=1), color="#D55E00" if ev.tipo == "calor" else "#0072B2", alpha=0.5, lw=0)
    eje.axhline(0, color="k", lw=0.6)
    eje.set_ylabel("anomalía (°C)")
    eje.set_title(f"{PUNTO_EJEMPLO}: anomalía de temperatura de cada observación y episodios (rojo cálido, azul frío)")
    eje.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

### E.2 Episodios de turbidez alta con los dos satélites

Un episodio de turbidez alta suele ser una **crecida** o un aporte extraordinario de sedimento. Si un episodio aparece
**con los dos satélites** a la vez en el mismo punto, es mucho más creíble que un artefacto de uno de ellos: es la
forma de **distinguir una anomalía real** de un error del sensor.

In [ ]:
if EJECUTAR_EXTREMOS:
    base_s2 = (int(FECHAS_S2[0][:4]), int(FECHAS_S2[1][:4]))
    # Sentinel-2 tiene pocos años: cada año se compara con los otros (dejar_fuera_anio) para que una crecida no infle su propia referencia
    ep_s2 = episodios(serie_s2, "b04", base_s2, brecha=12, tipos=("calor",), ventana=61, minimo_obs=4,
                      dejar_fuera_anio=True).assign(satelite="Sentinel-2")
    ep_l = episodios(serie_l_qc, "rojo", base_l, brecha=24, tipos=("calor",)).assign(satelite="Landsat")
    ep_turbidez = pd.concat([ep_s2, ep_l], ignore_index=True).replace({"tipo": {"calor": "turbidez alta"}})
    print(f"Episodios de turbidez alta: Sentinel-2 {len(ep_s2)}, Landsat {len(ep_l)}")

    # ¿Cuántos episodios de Sentinel-2 coinciden (±5 días) con uno de Landsat en el mismo punto?
    margen = pd.Timedelta(days=5)
    ep_s2 = ep_s2.copy()
    ep_s2["visto_por_landsat"] = [bool(((ep_l["punto"] == e.punto) & (ep_l["inicio"] <= e.fin + margen) &
                                        (ep_l["fin"] >= e.inicio - margen)).any()) for e in ep_s2.itertuples()]
    if len(ep_s2):
        print(f"De {len(ep_s2)} episodios de Sentinel-2, {int(ep_s2['visto_por_landsat'].sum())} también los ve Landsat "
              "(en el período en que los dos tienen datos).")
        display(ep_s2[["punto", "inicio", "fin", "n_obs", "intensidad_max", "visto_por_landsat"]].round(3))

### E.3 Isolation Forest: anomalías con varias variables a la vez (Landsat)

El umbral pregunta "¿esta temperatura supera lo que ocurre el 10 % de las veces en esta época?". Isolation Forest
pregunta "¿esta **combinación** de anomalías (temperatura, turbidez, agua) es poco común?". Se compara con el índice de
Jaccard (observaciones en común / unión): 0 = ninguna en común, 1 = idénticas.

In [ ]:
if EJECUTAR_EXTREMOS:
    variables = ["st_c", "rojo", "ndwi"]
    obs = serie_l_qc[serie_l_qc["id"] == PUNTO_EJEMPLO].set_index("fecha")[variables].astype(float)
    clima_vars = {v: climatologia_ventana(obs[v], base=base_l) for v in variables}
    bosque = aislamiento_forest_observaciones(obs, variables, clima_vars, contaminacion=0.1)
    umbral = anomalias_observaciones(obs["st_c"], clima_vars["st_c"])
    marca_umbral = umbral["excede_calor"] | umbral["excede_frio"]
    print(f"{PUNTO_EJEMPLO}: {int(bosque['atipico'].sum())} observaciones atípicas para Isolation Forest, "
          f"{int(marca_umbral.sum())} fuera de p10-p90 en temperatura")
    print("Comparación:", {k: round(v, 2) if isinstance(v, float) else v for k, v in solapamiento(marca_umbral, bosque["atipico"]).items()})

    fig, eje = plt.subplots(figsize=(13, 4))
    a = bosque["st_c"].dropna()
    eje.plot(a.index, a, "o", ms=3, color="k", label="anomalía de temperatura")
    raros = bosque[bosque["atipico"]]
    eje.scatter(raros.index, raros["st_c"], s=45, facecolors="none", edgecolors="#E69F00", lw=1.5, label="Isolation Forest (3 variables)")
    marcados = umbral[marca_umbral].dropna(subset=["valor"])
    eje.scatter(marcados.index, marcados["anomalia"], s=12, color="#D55E00", zorder=3, label="fuera de p10-p90")
    eje.axhline(0, color="k", lw=0.6)
    eje.set_ylabel("anomalía (°C)")
    eje.set_title("Umbral clásico frente a Isolation Forest")
    eje.legend(fontsize=8)
    eje.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    ep_temp.to_csv(SALIDAS / "episodios_temperatura_landsat.csv", index=False)
    ep_turbidez.to_csv(SALIDAS / "episodios_turbidez.csv", index=False)

## Cierre: qué aporta cada satélite

| Pregunta | Mejor satélite | Por qué |
|---|---|---|
| Ríos angostos (afluentes de 30 a 60 m) | Sentinel-2 | píxel de 10 m; en Landsat el píxel mezcla agua y orilla |
| Muchas fechas en un período corto | Sentinel-2 | revisita de 2 a 5 días |
| Climatología, tendencias y anomalías de varios años | Landsat | serie desde 2013 |
| Temperatura del agua | Landsat | única con banda térmica |
| Turbidez en el Biobío principal | los dos | coinciden el mismo día: se pueden combinar para tener más fechas |
| Confirmar un evento extremo | los dos | un episodio que ven ambos satélites es más creíble que uno solo |

**Límites:** solo hay datos con cielo despejado (las crecidas con lluvia quedan tapadas); la reflectancia roja es un
indicador de turbidez, no concentración (sin muestras no hay mg/L); la temperatura de Landsat es la de la piel del agua
(error de 1 K o más); con 2 años de Sentinel-2 la climatología de sus episodios es corta.

In [ ]:
print("Archivos generados en", SALIDAS)
for archivo in sorted(SALIDAS.iterdir()):
    print(f"  {archivo.name:45s} {archivo.stat().st_size / 1024:8.0f} kB")